# 01 - Data Preparation: Modern SAM2 + COLMAP Pipeline

This notebook handles data preparation for the SplatFields training pipeline.

**Runtime Environment:** Modern Colab (Default PyTorch 2.x, CUDA 12)

**Features:**
- Google Drive integration for data storage
- COLMAP sparse and dense reconstruction
- SAM2 mask generation
- Structured data output for training

**CRITICAL:** NO silent failures - all commands use verbose output for troubleshooting.

## 1. Setup & Drive Mount

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define project paths - CUSTOMIZE THESE FOR YOUR PROJECT
PROJECT_ROOT = "/content/drive/MyDrive/SplatFields_Project"
DATASET_PATH = f"{PROJECT_ROOT}/raw_data"

# Output paths
OUTPUT_ROOT = f"{PROJECT_ROOT}/ready_to_train_dataset"
COLMAP_OUTPUT = f"{PROJECT_ROOT}/colmap_output"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_PATH: {DATASET_PATH}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")

## 2. Install Dependencies

Installing SAM2 (Segment Anything 2) and pycolmap with verbose output.

In [ ]:
# Install SAM2 (Segment Anything 2)
!pip install sam2

# Install pycolmap (CUDA enabled) with verbose output
!pip install pycolmap

# Install additional dependencies
!pip install opencv-python-headless pillow numpy trimesh plyfile

print("\n=== Installation Complete ===")

In [ ]:
# Imports
import os
import json
import shutil
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

# Check if pycolmap imported successfully
try:
    import pycolmap
    print(f"pycolmap version: {pycolmap.__version__}")
except ImportError as e:
    print(f"ERROR: pycolmap import failed: {e}")

print("All imports successful!")

## 3. Input Validation

Validate that required folders and data exist in Google Drive.

In [ ]:
def validate_input_structure(dataset_path):
    """Validate the input dataset structure."""
    required_folders = ['sequences', 'extra_views']
    
    print(f"Validating input structure at: {dataset_path}")
    print("="*60)
    
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")
    
    # Check required folders
    for folder in required_folders:
        folder_path = os.path.join(dataset_path, folder)
        if os.path.exists(folder_path):
            items = os.listdir(folder_path)
            print(f"✓ Found '{folder}/' with {len(items)} items")
        else:
            print(f"✗ Missing '{folder}/' folder")
    
    # Validate sequences folder
    sequences_path = os.path.join(dataset_path, 'sequences')
    if os.path.exists(sequences_path):
        cameras = [d for d in os.listdir(sequences_path) 
                   if os.path.isdir(os.path.join(sequences_path, d))]
        print(f"\nFound {len(cameras)} camera sequences:")
        
        frame_counts = {}
        for cam in sorted(cameras):
            cam_path = os.path.join(sequences_path, cam)
            frames = [f for f in os.listdir(cam_path) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            frame_counts[cam] = len(frames)
            print(f"  - {cam}: {len(frames)} frames")
        
        # Assert frame counts match
        unique_counts = set(frame_counts.values())
        if len(unique_counts) > 1:
            print(f"\n⚠️  WARNING: Frame counts do not match across cameras!")
            print(f"   Unique counts: {unique_counts}")
        else:
            print(f"\n✓ All cameras have {list(unique_counts)[0]} frames")
        
        return cameras, frame_counts
    
    return [], {}

# Run validation
cameras, frame_counts = validate_input_structure(DATASET_PATH)

## 4. COLMAP Pipeline (Sparse + Dense)

Run COLMAP feature extraction, matching, sparse reconstruction, and dense reconstruction using pycolmap.

In [ ]:
def run_colmap_pipeline(images_path, output_path):
    """
    Run the complete COLMAP pipeline: Feature extraction, matching,
    sparse reconstruction, image undistortion, and dense reconstruction.
    
    Args:
        images_path: Path to input images (extra_views/t=0 frames)
        output_path: Path for COLMAP output
    """
    import pycolmap
    
    os.makedirs(output_path, exist_ok=True)
    sparse_path = os.path.join(output_path, 'sparse')
    dense_path = os.path.join(output_path, 'dense')
    undistorted_path = os.path.join(output_path, 'undistorted')
    database_path = os.path.join(output_path, 'database.db')
    
    os.makedirs(sparse_path, exist_ok=True)
    os.makedirs(dense_path, exist_ok=True)
    
    print("="*60)
    print("COLMAP PIPELINE")
    print("="*60)
    
    # Step 1: Feature Extraction
    print("\n[1/5] Feature Extraction...")
    pycolmap.extract_features(
        database_path=database_path,
        image_path=images_path,
        camera_mode=pycolmap.CameraMode.AUTO,
        verbose=True
    )
    print("✓ Feature extraction complete")
    
    # Step 2: Feature Matching
    print("\n[2/5] Feature Matching...")
    pycolmap.match_exhaustive(
        database_path=database_path,
        verbose=True
    )
    print("✓ Feature matching complete")
    
    # Step 3: Sparse Reconstruction (Mapper)
    print("\n[3/5] Sparse Reconstruction (Mapper)...")
    reconstructions = pycolmap.incremental_mapping(
        database_path=database_path,
        image_path=images_path,
        output_path=sparse_path,
        options=pycolmap.IncrementalPipelineOptions(),
        verbose=True
    )
    print(f"✓ Created {len(reconstructions)} reconstruction(s)")
    
    # Step 4: Image Undistortion (CRITICAL for dense reconstruction)
    print("\n[4/5] Image Undistortion...")
    pycolmap.undistort_images(
        output_path=undistorted_path,
        input_path=os.path.join(sparse_path, '0'),
        image_path=images_path,
        verbose=True
    )
    print("✓ Image undistortion complete")
    
    # Step 5: Dense Reconstruction (Patch Match Stereo)
    print("\n[5/5] Dense Reconstruction (Patch Match Stereo)...")
    pycolmap.patch_match_stereo(
        workspace_path=undistorted_path,
        verbose=True
    )
    
    # Fuse depth maps to create dense point cloud
    print("\n[5b] Stereo Fusion...")
    pycolmap.stereo_fusion(
        output_path=os.path.join(dense_path, 'dense.ply'),
        workspace_path=undistorted_path,
        verbose=True
    )
    print("✓ Dense reconstruction complete")
    
    print("\n" + "="*60)
    print("COLMAP PIPELINE COMPLETE")
    print("="*60)
    
    return {
        'sparse_path': sparse_path,
        'dense_path': dense_path,
        'undistorted_path': undistorted_path,
        'database_path': database_path
    }

# Run COLMAP on extra_views (t=0 frames)
extra_views_path = os.path.join(DATASET_PATH, 'extra_views')
if os.path.exists(extra_views_path):
    colmap_result = run_colmap_pipeline(extra_views_path, COLMAP_OUTPUT)
else:
    print(f"ERROR: extra_views folder not found at {extra_views_path}")
    print("Please ensure your data is properly structured.")

## 5. SAM2 Mask Generation

Generate segmentation masks using SAM2 (Segment Anything 2) for each camera sequence.

In [ ]:
# Load SAM2 configuration (user-provided prompts)
CONFIG_PATH = f"{PROJECT_ROOT}/config/sam2_prompts.json"

def load_sam2_config(config_path):
    """Load SAM2 prompts from configuration file."""
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = json.load(f)
        print(f"✓ Loaded SAM2 config from {config_path}")
        return config
    else:
        print(f"⚠️  SAM2 config not found at {config_path}")
        print("Using default configuration (center point prompt)")
        return {
            'prompts': [
                {'type': 'point', 'coords': [[0.5, 0.5]], 'labels': [1]}
            ]
        }

sam2_config = load_sam2_config(CONFIG_PATH)

In [ ]:
def run_sam2_masking(sequences_path, output_masks_path, config):
    """
    Run SAM2 mask generation on all camera sequences.
    
    Args:
        sequences_path: Path to camera sequence folders
        output_masks_path: Output path for generated masks
        config: SAM2 configuration with prompts
    """
    try:
        from sam2.build_sam import build_sam2_video_predictor
        from sam2.sam2_video_predictor import SAM2VideoPredictor
    except ImportError:
        print("ERROR: SAM2 not properly installed. Please run the installation cell.")
        return
    
    os.makedirs(output_masks_path, exist_ok=True)
    
    print("="*60)
    print("SAM2 MASK GENERATION")
    print("="*60)
    
    # Load SAM2 model (sam2_hiera_large)
    print("\nLoading SAM2 model (sam2_hiera_large)...")
    
    # Check for model checkpoint
    checkpoint_path = "sam2_hiera_large.pt"
    config_path = "sam2_hiera_l.yaml"
    
    # Download checkpoint if needed
    if not os.path.exists(checkpoint_path):
        print("Downloading SAM2 checkpoint...")
        !wget --progress=bar:force https://dl.fbaipublicfiles.com/segment_anything_2/sam2_hiera_large.pt
    
    try:
        predictor = build_sam2_video_predictor(
            config_file=config_path,
            ckpt_path=checkpoint_path,
        )
        print("✓ SAM2 model loaded")
    except Exception as e:
        print(f"Error loading SAM2 with video predictor: {e}")
        print("Falling back to alternative SAM2 loading method...")
        try:
            # Alternative loading method
            from sam2 import sam_model_registry
            predictor = sam_model_registry["sam2_hiera_l"](checkpoint=checkpoint_path)
            print("✓ SAM2 model loaded via alternative method")
        except Exception as e2:
            print(f"ERROR: Failed to load SAM2 model: {e2}")
            print("\nTroubleshooting steps:")
            print("1. Ensure SAM2 is installed: pip install sam2")
            print("2. Check if checkpoint downloaded: ls -la sam2_hiera_large.pt")
            print("3. Try reinstalling: pip uninstall sam2 && pip install sam2")
            raise RuntimeError("SAM2 model loading failed. See troubleshooting steps above.")
    
    # Get camera directories
    cameras = sorted([d for d in os.listdir(sequences_path) 
                      if os.path.isdir(os.path.join(sequences_path, d))])
    
    print(f"\nProcessing {len(cameras)} camera sequences...")
    
    for cam_idx, cam_name in enumerate(cameras):
        print(f"\n[{cam_idx+1}/{len(cameras)}] Processing {cam_name}...")
        
        cam_path = os.path.join(sequences_path, cam_name)
        cam_mask_path = os.path.join(output_masks_path, cam_name)
        os.makedirs(cam_mask_path, exist_ok=True)
        
        # Get sorted frame files
        frames = sorted([f for f in os.listdir(cam_path) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        
        if not frames:
            print(f"  ⚠️  No frames found in {cam_name}")
            continue
        
        # Initialize video state
        inference_state = predictor.init_state(video_path=cam_path)
        
        # Add prompts from config to the first frame
        prompts = config.get('prompts', [])
        for prompt in prompts:
            if prompt['type'] == 'point':
                # Scale normalized coordinates to image size
                first_frame = Image.open(os.path.join(cam_path, frames[0]))
                w, h = first_frame.size
                
                points = np.array(prompt['coords']) * np.array([w, h])
                labels = np.array(prompt['labels'])
                
                _, _, out_mask_logits = predictor.add_new_points_or_box(
                    inference_state=inference_state,
                    frame_idx=0,
                    obj_id=1,
                    points=points,
                    labels=labels,
                )
        
        # Propagate masks through the video
        print(f"  Propagating masks through {len(frames)} frames...")
        video_segments = {}
        
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
            video_segments[out_frame_idx] = {
                out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy()
                for i, out_obj_id in enumerate(out_obj_ids)
            }
        
        # Save masks
        for frame_idx, frame_name in enumerate(frames):
            if frame_idx in video_segments:
                mask = video_segments[frame_idx].get(1, np.zeros((h, w), dtype=np.uint8))
                mask = (mask.squeeze() * 255).astype(np.uint8)
            else:
                mask = np.zeros((h, w), dtype=np.uint8)
            
            mask_filename = os.path.splitext(frame_name)[0] + '.png'
            cv2.imwrite(os.path.join(cam_mask_path, mask_filename), mask)
        
        print(f"  ✓ Saved {len(frames)} masks to {cam_mask_path}")
        
        # Reset state for next camera
        predictor.reset_state(inference_state)
    
    print("\n" + "="*60)
    print("SAM2 MASK GENERATION COMPLETE")
    print("="*60)

# Run SAM2 masking
sequences_path = os.path.join(DATASET_PATH, 'sequences')
masks_output_path = os.path.join(OUTPUT_ROOT, 'masks')

if os.path.exists(sequences_path):
    run_sam2_masking(sequences_path, masks_output_path, sam2_config)
else:
    print(f"ERROR: sequences folder not found at {sequences_path}")

## 6. Final Data Structuring (The Handoff)

Organize all outputs into the `ready_to_train_dataset/` structure for training notebook.

In [ ]:
def structure_training_data(colmap_result, sequences_path, masks_path, output_root):
    """
    Structure all data for the training pipeline with hierarchical directory structure.
    
    Output structure:
    ready_to_train_dataset/
    ├── images/
    │   ├── cam01/       # Hierarchical structure per camera
    │   │   ├── frame_0001.png
    │   │   └── ...
    │   └── cam02/
    ├── masks/
    │   ├── cam01/       # Matching hierarchical structure
    │   │   ├── frame_0001.png
    │   │   └── ...
    │   └── cam02/
    ├── sparse/0/       # cameras.bin, images.bin from undistorted output
    ├── init_ply/       # dense.ply
    └── metadata.json   # List of training camera names
    """
    print("="*60)
    print("STRUCTURING TRAINING DATA")
    print("="*60)
    
    # Create output directories
    images_output = os.path.join(output_root, 'images')
    masks_output = os.path.join(output_root, 'masks')
    sparse_output = os.path.join(output_root, 'sparse', '0')
    init_ply_output = os.path.join(output_root, 'init_ply')
    
    for path in [images_output, masks_output, sparse_output, init_ply_output]:
        os.makedirs(path, exist_ok=True)
    
    # 1. Copy undistorted images (to match dense.ply coordinate frame)
    print("\n[1/4] Copying undistorted images...")
    undistorted_images = os.path.join(colmap_result['undistorted_path'], 'images')
    if os.path.exists(undistorted_images):
        for img in os.listdir(undistorted_images):
            shutil.copy2(
                os.path.join(undistorted_images, img),
                os.path.join(images_output, img)
            )
        print(f"  ✓ Copied {len(os.listdir(undistorted_images))} images")
    else:
        print(f"  ⚠️  Undistorted images not found, copying from sequences with hierarchical structure...")
        # Fallback: copy from original sequences maintaining hierarchical structure
        total_copied = 0
        if os.path.exists(sequences_path):
            for cam_folder in os.listdir(sequences_path):
                cam_src_path = os.path.join(sequences_path, cam_folder)
                if os.path.isdir(cam_src_path):
                    # Create camera subdirectory in images output
                    cam_dst_path = os.path.join(images_output, cam_folder)
                    os.makedirs(cam_dst_path, exist_ok=True)
                    
                    for img in os.listdir(cam_src_path):
                        if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                            shutil.copy2(
                                os.path.join(cam_src_path, img),
                                os.path.join(cam_dst_path, img)  # Preserve original filename in subdirectory
                            )
                            total_copied += 1
        print(f"  ✓ Copied {total_copied} images with hierarchical structure")
    
    # 2. Copy/link masks with hierarchical structure
    print("\n[2/4] Copying masks with hierarchical structure...")
    if os.path.exists(masks_path) and masks_path != masks_output:
        if os.path.exists(masks_output):
            shutil.rmtree(masks_output)
        shutil.copytree(masks_path, masks_output)
        print(f"  ✓ Copied masks (preserving hierarchical structure)")
    else:
        print(f"  ✓ Masks already in place")
    
    # 3. Copy sparse model from UNDISTORTED output (IMPORTANT!)
    print("\n[3/4] Copying sparse model from undistorted output...")
    undistorted_sparse = os.path.join(colmap_result['undistorted_path'], 'sparse')
    if os.path.exists(undistorted_sparse):
        for file in ['cameras.bin', 'images.bin', 'points3D.bin']:
            src = os.path.join(undistorted_sparse, file)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(sparse_output, file))
                print(f"  ✓ Copied {file}")
    else:
        # Fallback to original sparse
        print("  ⚠️  Undistorted sparse not found, copying from original sparse...")
        original_sparse = os.path.join(colmap_result['sparse_path'], '0')
        if os.path.exists(original_sparse):
            for file in ['cameras.bin', 'images.bin', 'points3D.bin']:
                src = os.path.join(original_sparse, file)
                if os.path.exists(src):
                    shutil.copy2(src, os.path.join(sparse_output, file))
                    print(f"  ✓ Copied {file}")
    
    # 4. Copy dense point cloud
    print("\n[4/4] Copying dense point cloud...")
    dense_ply_src = os.path.join(colmap_result['dense_path'], 'dense.ply')
    dense_ply_dst = os.path.join(init_ply_output, 'dense.ply')
    if os.path.exists(dense_ply_src):
        shutil.copy2(dense_ply_src, dense_ply_dst)
        print(f"  ✓ Copied dense.ply")
    else:
        print(f"  ⚠️  dense.ply not found at {dense_ply_src}")
    
    # 5. Create metadata.json with training camera names
    print("\n[5/4] Creating metadata.json...")
    
    # Get camera names from the sequences folder
    train_cam_names = []
    if os.path.exists(sequences_path):
        train_cam_names = sorted([d for d in os.listdir(sequences_path) 
                                  if os.path.isdir(os.path.join(sequences_path, d))])
    
    metadata = {
        'train_cam_names': train_cam_names,
        'num_cameras': len(train_cam_names),
        'description': 'Training dataset prepared by 01_DataPrep_Modern_SAM2_Colmap.ipynb'
    }
    
    metadata_path = os.path.join(output_root, 'metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"  ✓ Created metadata.json with {len(train_cam_names)} cameras")
    print(f"  Cameras: {train_cam_names}")
    
    print("\n" + "="*60)
    print("DATA STRUCTURING COMPLETE")
    print("="*60)
    print(f"\nOutput directory: {output_root}")
    print("\nDirectory structure:")
    for root, dirs, files in os.walk(output_root):
        level = root.replace(output_root, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = '  ' * (level + 1)
        for file in files[:5]:  # Limit files shown
            print(f"{sub_indent}{file}")
        if len(files) > 5:
            print(f"{sub_indent}... and {len(files) - 5} more files")
    
    return metadata

# Structure the training data
sequences_path = os.path.join(DATASET_PATH, 'sequences')
masks_path = os.path.join(OUTPUT_ROOT, 'masks')

if 'colmap_result' in dir():
    metadata = structure_training_data(
        colmap_result, 
        sequences_path, 
        masks_path, 
        OUTPUT_ROOT
    )
else:
    print("ERROR: COLMAP pipeline was not run. Please run the COLMAP cell first.")

## 7. Verification & Summary

In [ ]:
def verify_output(output_root):
    """Verify the output dataset structure is complete."""
    print("="*60)
    print("VERIFICATION")
    print("="*60)
    
    required_items = [
        ('images/', 'directory'),
        ('masks/', 'directory'),
        ('sparse/0/', 'directory'),
        ('sparse/0/cameras.bin', 'file'),
        ('sparse/0/images.bin', 'file'),
        ('init_ply/', 'directory'),
        ('init_ply/dense.ply', 'file'),
        ('metadata.json', 'file'),
    ]
    
    all_ok = True
    for item, item_type in required_items:
        path = os.path.join(output_root, item)
        if item_type == 'directory':
            exists = os.path.isdir(path)
        else:
            exists = os.path.isfile(path)
        
        status = "✓" if exists else "✗"
        print(f"  {status} {item}")
        if not exists:
            all_ok = False
    
    if all_ok:
        print("\n✓ All required items present!")
        print(f"\n📁 Dataset ready at: {output_root}")
        print("\nNext step: Run 02_Train_Legacy_SplatFields.ipynb")
    else:
        print("\n⚠️  Some items are missing. Please check the pipeline output.")
    
    return all_ok

verify_output(OUTPUT_ROOT)

In [ ]:
# Final summary
print("\n" + "="*60)
print("DATA PREPARATION COMPLETE")
print("="*60)
print(f"\nDataset location: {OUTPUT_ROOT}")
print("\nTo proceed with training:")
print("1. Open 02_Train_Legacy_SplatFields.ipynb")
print("2. Update PROJECT_ROOT to match this notebook")
print("3. Run all cells to train the model")
print("\n" + "="*60)